<a href="https://colab.research.google.com/github/KULDEEPSONI-source/DEEPLEARNING/blob/main/keras_hyperparameter_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [222]:
import numpy as np
import pandas as pd


In [223]:
df = pd.read_csv('diabetes.csv')

In [224]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [225]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


X = df.iloc[:, :-1]   # all columns except last → features
y = df.iloc[:, -1]    # only the last column → target/label

In [226]:
x=df.iloc[:,:8].values
y=df.iloc[:,8].values

In [227]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X= scaler.fit_transform(x)
#

In [228]:
X.shape

(768, 8)

In [229]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [230]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense


In [231]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer = 'Adam', loss= 'binary_crossentropy' , metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [232]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6645 - loss: 0.6555 - val_accuracy: 0.6429 - val_loss: 0.6336
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6726 - loss: 0.5981 - val_accuracy: 0.6753 - val_loss: 0.5814
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6922 - loss: 0.5528 - val_accuracy: 0.7078 - val_loss: 0.5433
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7345 - loss: 0.5210 - val_accuracy: 0.7403 - val_loss: 0.5172
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7573 - loss: 0.4998 - val_accuracy: 0.7468 - val_loss: 0.4984
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7720 - loss: 0.4838 - val_accuracy: 0.7662 - val_loss: 0.4857
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4720 - val_accuracy: 0.7727 - val_loss: 0.4784
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7752 - loss: 0.4643 - val_accuracy: 0.7857 - 

In [233]:
# 1.how to select appropriate optimizer
# 2. NO, of nodes in a layer
# 3. how to select no. of hidden layer

In [234]:
# !pip install keras-tuner

In [235]:
import kerastuner as kt

In [236]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop','adadelta'])
  model.compile(optimizer = optimizer, loss= 'binary_crossentropy' , metrics=['accuracy'])
  return model

In [237]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [238]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [239]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [240]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [241]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [242]:
model.fit(X_train, y_train, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7492 - loss: 0.5291 - val_accuracy: 0.7987 - val_loss: 0.5092
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7524 - loss: 0.5153 - val_accuracy: 0.7792 - val_loss: 0.4992
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7541 - loss: 0.5048 - val_accuracy: 0.7922 - val_loss: 0.4926
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7622 - loss: 0.4968 - val_accuracy: 0.7857 - val_loss: 0.4857
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.4897 - val_accuracy: 0.7792 - val_loss: 0.4801
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.4836 - val_accuracy: 0.7857 - val_loss: 0.4760
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4785 - val_accuracy: 0.7792 - val_loss: 0.4721
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4750 - val_accuracy: 0.79

# 2. NO, of nodes in a layer

In [243]:
def build_model(hp):
  model = Sequential()
  units = hp.Int('units', min_value=8, max_value=64, step=8)
  model.add(Dense(units, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop','adadelta'])
  model.compile(optimizer = optimizer, loss= 'binary_crossentropy' , metrics=['accuracy'])
  return model
#

In [244]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory = ' mydir')

Reloading Tuner from  mydir/untitled_project/tuner0.json


In [245]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [246]:
tuner.get_best_hyperparameters()[0].values

{'units': 48, 'optimizer': 'adam'}

In [247]:
model = tuner.get_best_models(num_models=1)[0]

In [248]:
model.fit(X_train, y_train, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7655 - loss: 0.5266 - val_accuracy: 0.7987 - val_loss: 0.4986
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.5047 - val_accuracy: 0.7987 - val_loss: 0.4818
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7687 - loss: 0.4912 - val_accuracy: 0.7857 - val_loss: 0.4708
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4798 - val_accuracy: 0.7792 - val_loss: 0.4647
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4729 - val_accuracy: 0.7792 - val_loss: 0.4599
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4678 - val_accuracy: 0.7857 - val_loss: 0.4567
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4629 - val_accuracy: 0.7987 - val_loss: 0.4539
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7850 - loss: 0.4591 - val_accuracy: 0.79

# 3. how to select no. of hidden layer

In [249]:
def build_model(hp):
  model = Sequential()
  model.add(Dense(72, activation='relu', input_dim=8))
  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    model.add(Dense(72, activation='relu'))
  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

In [250]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials=3,directory = 'my_dt')

Reloading Tuner from my_dt/untitled_project/tuner0.json


In [251]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [252]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [253]:
model.fit(X_train, y_train, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8274 - loss: 0.3995 - val_accuracy: 0.8052 - val_loss: 0.4540
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8241 - loss: 0.3990 - val_accuracy: 0.8117 - val_loss: 0.4529
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8257 - loss: 0.3994 - val_accuracy: 0.8117 - val_loss: 0.4532
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8306 - loss: 0.4004 - val_accuracy: 0.8182 - val_loss: 0.4557
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8306 - loss: 0.3983 - val_accuracy: 0.8117 - val_loss: 0.4557
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8306 - loss: 0.3980 - val_accuracy: 0.8117 - val_loss: 0.4548
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8371 - loss: 0.3982 - val_accuracy: 0.8052 - val_loss: 0.4581
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8306 - loss: 0.3973 - val_accuracy: 0.805

In [254]:
def build_model(hp):
  model = Sequential()
  # model.add(Dense(72, activation='relu', input_dim=8))
  counter =0
  num_layers = hp.Int('num_layers', min_value=1, max_value = 10)
  for i in range(num_layers):
    if counter == 0:
      model.add(Dense(
          hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
          activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
          input_dim=8
      ))

    else:
      model.add(Dense(
          hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
          activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])
      ))
    counter+=1
  model.add(Dense(1, activation='sigmoid'))
  model.compile(optimizer=hp.Choice('optimizer', values=['rmsprop','adam','sgd', 'nadam','adadelta']),
                loss='binary_crossentropy',
                metrics=['accuracy']
                )
  return model

In [255]:
tuner = kt.RandomSearch(build_model,
                        objective = 'val_accuracy',
                        max_trials=3,directory = 'all')

Reloading Tuner from all/untitled_project/tuner0.json


In [256]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 48,
 'activation0': 'tanh',
 'optimizer': 'nadam',
 'units1': 40,
 'activation1': 'tanh',
 'units2': 96,
 'activation2': 'tanh',
 'units3': 120,
 'activation3': 'sigmoid',
 'units4': 120,
 'activation4': 'tanh',
 'units5': 88,
 'activation5': 'sigmoid',
 'units6': 40,
 'activation6': 'tanh'}

In [257]:
tuner.get_best_models()[0].summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 19 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 48)             │           432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 40)             │         1,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 96)             │         3,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            97 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,425 (25.10 KB)

 Trainable params: 6,425 (25.10 KB)

 Non-trainable params: 0 (0.00 B)

In [258]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [259]:
model = tuner.get_best_models(num_models=1)[0]

In [260]:
model.fit(X_train, y_train, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.7459 - loss: 0.4879 - val_accuracy: 0.7922 - val_loss: 0.4573
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7671 - loss: 0.4712 - val_accuracy: 0.7922 - val_loss: 0.4535
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7736 - loss: 0.4665 - val_accuracy: 0.7792 - val_loss: 0.4528
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7720 - loss: 0.4633 - val_accuracy: 0.8052 - val_loss: 0.4538
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7720 - loss: 0.4625 - val_accuracy: 0.7857 - val_loss: 0.4566
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7687 - loss: 0.4622 - val_accuracy: 0.7727 - val_loss: 0.4599
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4584 - val_accuracy: 0.7727 - val_loss: 0.4651
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4639 - val_accuracy: